#Sistema concurrente de predicción de depresión posparto con K-means en Go

# **Objetivo:** Procesar PPD dataset con máxima eficiencia


##Dependencias, utilizamos la API de Kaggle

In [3]:
# Instalar librerías necesarias
!pip install -q polars pyarrow kaggle dask requests psutil

print("Correcto")

Correcto


In [4]:
from google.colab import files
import os
from pathlib import Path

print("CONFIGURAR KAGGLE API")

#directorio .kaggle
kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)

print("\n1️archivo kaggle.json")
print("Descárgalo de: https://www.kaggle.com/settings/account)")
print("\n📁Selecciona kaggle.json...")

uploaded_files = files.upload()

#ubicación correcta
for filename in uploaded_files.keys():
    if filename == 'kaggle.json':
        os.rename(filename, str(kaggle_dir / 'kaggle.json'))
        os.chmod(str(kaggle_dir / 'kaggle.json'), 0o600)
        print(f"kaggle.json configurado correctamente")
    else:
        os.remove(filename)

print(f"\n📍 Ruta: {kaggle_dir / 'kaggle.json'}")

CONFIGURAR KAGGLE API

1️archivo kaggle.json
Descárgalo de: https://www.kaggle.com/settings/account)

📁Selecciona kaggle.json...


Saving kaggle.json to kaggle.json
kaggle.json configurado correctamente

📍 Ruta: /root/.kaggle/kaggle.json


In [5]:
import subprocess
import os

print("DESCARGAR DATASET DESDE KAGGLE")

#directorio para dataset
os.makedirs('data', exist_ok=True)

print("\Descargando dataset...")
print("(Sin abrir navegador, sin Google Drive)\n")

# Descargar usando Kaggle CLI
result = subprocess.run(
    ['kaggle', 'datasets', 'download', '-d',
     'parvezalmuqtadir2348/postpartum-depression',
     '-p', 'data', '--unzip'],
    capture_output=True,
    text=True
)
if result.returncode == 0:
    print("Descargado exitosamente")
    # Listar archivos descargados
    files_list = os.listdir('data')
    print(f"\nArchivos descargados: {len(files_list)}")
    for f in files_list[:5]:
        size = os.path.getsize(f'data/{f}') / (1024**2)
        print(f"   • {f} ({size:.1f} MB)")
else:
    print(f"Error: {result.stderr}")

<>:9: SyntaxWarning: invalid escape sequence '\D'
<>:9: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_592/2182593382.py:9: SyntaxWarning: invalid escape sequence '\D'
  print("\Descargando dataset...")


DESCARGAR DATASET DESDE KAGGLE
\Descargando dataset...
(Sin abrir navegador, sin Google Drive)

Descargado exitosamente

Archivos descargados: 1
   • post natal data.csv (0.1 MB)


##Cargar con Polars (Lazy Evaluation)

In [6]:
import polars as pl
from polars import col, when
import psutil

print("CARGAR DATASET CON POLARS")

# Encontrar el CSV
csv_files = [f for f in os.listdir('data') if f.endswith('.csv')]
csv_path = f'data/{csv_files[0]}'

print(f"\nArchivo: {csv_files[0]}")
print(f"Tamaño: {os.path.getsize(csv_path) / (1024**2):.1f} MB")

print("\nOPCIÓN A: scan_csv() - LAZY (SIN cargar en memoria)")
start_memory = psutil.Process().memory_info().rss / (1024**2)
print(f"RAM antes: {start_memory:.1f} MB")

# Lazy: solo lee headers, crea plan de ejecución
df_lazy = pl.scan_csv(csv_path)

current_memory = psutil.Process().memory_info().rss / (1024**2)
print(f"RAM después: {current_memory:.1f} MB")
print(f"✓ Datos NO cargados en memoria")
print(f"✓ Solo plan de ejecución ({(current_memory - start_memory):.1f} MB)")

print("\nINFORMACIÓN DEL SCHEMA:")
print(df_lazy.schema)

CARGAR DATASET CON POLARS

Archivo: post natal data.csv
Tamaño: 0.1 MB

OPCIÓN A: scan_csv() - LAZY (SIN cargar en memoria)
RAM antes: 277.7 MB
RAM después: 278.6 MB
✓ Datos NO cargados en memoria
✓ Solo plan de ejecución (0.9 MB)

INFORMACIÓN DEL SCHEMA:
Schema({'Timestamp': String, 'Age': String, 'Feeling sad or Tearful': String, 'Irritable towards baby & partner': String, 'Trouble sleeping at night': String, 'Problems concentrating or making decision': String, 'Overeating or loss of appetite': String, 'Feeling anxious': String, 'Feeling of guilt': String, 'Problems of bonding with baby': String, 'Suicide attempt': String})


/tmp/ipykernel_592/3621627457.py:27: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  print(df_lazy.schema)


##Autodetectar estructura

In [11]:
print("DETECTANDO ESTRUCTURA DEL DATASET")

# Clasificar columnas por tipo
schema = df_lazy.schema
numeric_cols = [name for name, dtype in schema.items()
                if str(dtype) in ['Int64', 'Int32', 'Float64', 'Float32']]
string_cols = [name for name, dtype in schema.items()
              if str(dtype) == 'String']

print(f"\n✓ Columnas numéricas: {len(numeric_cols)}")
for col in numeric_cols:
    print(f"   • {col}")

print(f"\n✓ Columnas de texto (categóricas): {len(string_cols)}")
for col in string_cols:
    print(f"   • {col}")

# Detectar variable target (columna con menos valores únicos)
print("\nDETECTANDO VARIABLE TARGET...")
df_sample = df_lazy.limit(1000).collect()

target_col = None
for col in string_cols:
    unique_count = df_sample[col].n_unique()
    if unique_count <= 3:  # Probablemente binaria/ternaria
        target_col = col
        print(f"   ✅ TARGET: '{col}' ({unique_count} valores únicos)")
        break

if not target_col:
    target_col = string_cols[-1] if string_cols else None
    print(f"   → Usando última columna: {target_col}")


DETECTANDO ESTRUCTURA DEL DATASET

✓ Columnas numéricas: 0

✓ Columnas de texto (categóricas): 11
   • Timestamp
   • Age
   • Feeling sad or Tearful
   • Irritable towards baby & partner
   • Trouble sleeping at night
   • Problems concentrating or making decision
   • Overeating or loss of appetite
   • Feeling anxious
   • Feeling of guilt
   • Problems of bonding with baby
   • Suicide attempt

DETECTANDO VARIABLE TARGET...
   ✅ TARGET: 'Feeling sad or Tearful' (3 valores únicos)


/tmp/ipykernel_592/3845015200.py:4: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  schema = df_lazy.schema


##Transformaciones Lazy (SIN ejecutar aún)

In [13]:
import polars as pl
from polars import col, when

print("DEFINIR TRANSFORMACIONES LAZY (aún NO se ejecutan)")


df_transformed = df_lazy

print("\n1CONVERSIÓN DE TIPOS: STRING → NÚMERO")
print("   (Age: '35-40' → 37.5, respuestas: Yes→1, No→0, etc.)")

# Procesar Age si es string de rango
if 'Age' in string_cols:
    print("   • Convertir Age (rango) a número")
    df_transformed = df_transformed.with_columns(
        when(col('Age') == '18-20').then(19)
        .when(col('Age') == '20-25').then(22.5)
        .when(col('Age') == '25-30').then(27.5)
        .when(col('Age') == '30-35').then(32.5)
        .when(col('Age') == '35-40').then(37.5)
        .when(col('Age') == '40-45').then(42.5)
        .when(col('Age') == '45-50').then(47.5)
        .when(col('Age') == '50-55').then(52.5)
        .otherwise(35)  # Default
        .cast(pl.Float64)
        .alias('Age_numeric')
    )

# Codificar respuestas binarias/ternarias
print("   • Codificar respuestas (Yes→1, No→0, Sometimes→0.5, Maybe→0.5)")
for col_name in string_cols:
    if col_name not in ['Timestamp', 'Age']:
        df_transformed = df_transformed.with_columns(
            when(col(col_name) == 'Yes').then(1)
            .when(col(col_name) == 'No').then(0)
            .when(col(col_name) == 'Sometimes').then(0.5)
            .when(col(col_name) == 'Maybe').then(0.5)
            .when(col(col_name) == 'Not interested to say').then(0)  # Para Suicide attempt
            .otherwise(0)  # Default
            .cast(pl.Float64)
            .alias(f"{col_name}_encoded")
        )

print("\n2️IMPUTACIÓN: Llenar valores faltantes")
print("   (Usar mediana de cada columna)")

df_transformed = df_transformed.with_columns([
    when(col('Age_numeric').is_null())
        .then(col('Age_numeric').median())
        .otherwise(col('Age_numeric'))
        .alias('Age_numeric'),
])

for col_name in string_cols:
    if col_name not in ['Timestamp', 'Age']:
        df_transformed = df_transformed.with_columns(
            when(col(f"{col_name}_encoded").is_null())
                .then(col(f"{col_name}_encoded").median())
                .otherwise(col(f"{col_name}_encoded"))
                .alias(f"{col_name}_encoded")
        )

print("\n3️NORMALIZACIÓN: Z-score (media=0, std=1)")
print("   (Crítico para K-means: Age 18-50 vs síntomas 0-1)")

df_transformed = df_transformed.with_columns([
    ((col('Age_numeric') - col('Age_numeric').mean()) / col('Age_numeric').std())
        .alias('Age_normalized'),
])

for col_name in string_cols:
    if col_name not in ['Timestamp', 'Age']:
        df_transformed = df_transformed.with_columns(
            ((col(f"{col_name}_encoded") - col(f"{col_name}_encoded").mean()) / col(f"{col_name}_encoded").std())
                .alias(f"{col_name}_normalized")
        )

print("\n4️VALIDACIÓN: Filtrar registros inválidos")
print("   (Age entre 18-50 años, sin valores extremos)")

df_transformed = df_transformed.filter(
    (col('Age_numeric') >= 18) & (col('Age_numeric') <= 50)
)

print("\nSELECCIONAR COLUMNAS FINALES")
print("   (Solo features normalizadas + target)")

# Seleccionar columnas: features normalizadas + target
final_cols = [c for c in df_transformed.collect().columns
              if c.endswith('_normalized')] + [f"{target_col}_encoded" if target_col in string_cols else target_col]
df_transformed = df_transformed.select(final_cols)

print(f"   ✓ Features seleccionadas: {len(final_cols)}")
print("   ✓ Ready para ejecutar")
print("Las transformaciones están DEFINIDAS pero NO ejecutadas")
print("  El optimizador de Polars las ejecutará en paralelo al .collect()")

DEFINIR TRANSFORMACIONES LAZY (aún NO se ejecutan)

1CONVERSIÓN DE TIPOS: STRING → NÚMERO
   (Age: '35-40' → 37.5, respuestas: Yes→1, No→0, etc.)
   • Convertir Age (rango) a número
   • Codificar respuestas (Yes→1, No→0, Sometimes→0.5, Maybe→0.5)

2️IMPUTACIÓN: Llenar valores faltantes
   (Usar mediana de cada columna)

3️NORMALIZACIÓN: Z-score (media=0, std=1)
   (Crítico para K-means: Age 18-50 vs síntomas 0-1)

4️VALIDACIÓN: Filtrar registros inválidos
   (Age entre 18-50 años, sin valores extremos)

SELECCIONAR COLUMNAS FINALES
   (Solo features normalizadas + target)
   ✓ Features seleccionadas: 11
   ✓ Ready para ejecutar
Las transformaciones están DEFINIDAS pero NO ejecutadas
  El optimizador de Polars las ejecutará en paralelo al .collect()


##Ejecutar transformaciones (.collect())



In [14]:
import time

print("\EJECUTAR PLAN OPTIMIZADO")
print("\Procesando...")
start_time = time.time()
start_memory = psutil.Process().memory_info().rss / (1024**2)

# .collect() ejecuta el plan optimizado
try:
    df_processed = df_transformed.collect()

    elapsed_time = time.time() - start_time
    end_memory = psutil.Process().memory_info().rss / (1024**2)
    peak_memory = end_memory - start_memory

    print(f"\Ejecución completada")
    print(f"Tiempo: {elapsed_time:.2f} segundos")
    print(f"RAM pico: {peak_memory:.1f} MB")
    print(f"Registros procesados: {len(df_processed):,}")
    print(f"Features: {df_processed.shape[1]}")

    print("\Vista previa:")
    print(df_processed.head())

except Exception as e:
    print(f"Error durante ejecución: {e}")
    print("Posible solución:")
    print("Verifica que las columnas existan en el dataset")
    print("Revisa los nombres de columnas en el PASO 4")

\EJECUTAR PLAN OPTIMIZADO
\Procesando...
\Ejecución completada
Tiempo: 0.01 segundos
RAM pico: 0.9 MB
Registros procesados: 1,503
Features: 11
\Vista previa:
shape: (5, 11)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ Age_norma ┆ Feeling   ┆ Irritable ┆ Trouble   ┆ … ┆ Feeling   ┆ Problems  ┆ Suicide   ┆ Feeling  │
│ lized     ┆ sad or    ┆ towards   ┆ sleeping  ┆   ┆ of guilt_ ┆ of        ┆ attempt_n ┆ sad or   │
│ ---       ┆ Tearful_n ┆ baby &    ┆ at night_ ┆   ┆ normalize ┆ bonding   ┆ ormalized ┆ Tearful_ │
│ f64       ┆ ormali…   ┆ partn…    ┆ norm…     ┆   ┆ d         ┆ with      ┆ ---       ┆ encoded  │
│           ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ baby_…    ┆ f64       ┆ ---      │
│           ┆ f64       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ ---       ┆           ┆ f64      │
│           ┆           ┆           ┆           ┆   ┆           ┆ f64       ┆           ┆          │
╞═══════════╪══════

<>:3: SyntaxWarning: invalid escape sequence '\E'
<>:4: SyntaxWarning: invalid escape sequence '\P'
<>:16: SyntaxWarning: invalid escape sequence '\E'
<>:22: SyntaxWarning: invalid escape sequence '\V'
<>:3: SyntaxWarning: invalid escape sequence '\E'
<>:4: SyntaxWarning: invalid escape sequence '\P'
<>:16: SyntaxWarning: invalid escape sequence '\E'
<>:22: SyntaxWarning: invalid escape sequence '\V'
/tmp/ipykernel_592/1074025940.py:3: SyntaxWarning: invalid escape sequence '\E'
  print("\EJECUTAR PLAN OPTIMIZADO")
/tmp/ipykernel_592/1074025940.py:4: SyntaxWarning: invalid escape sequence '\P'
  print("\Procesando...")
/tmp/ipykernel_592/1074025940.py:16: SyntaxWarning: invalid escape sequence '\E'
  print(f"\Ejecución completada")
/tmp/ipykernel_592/1074025940.py:22: SyntaxWarning: invalid escape sequence '\V'
  print("\Vista previa:")


In [15]:
print("VALIDACIÓN DE CALIDAD")

print(f"\n✓ Registros originales: {len(df_lazy.collect()):,}")
print(f"✓ Registros después limpieza: {len(df_processed):,}")
print(f"✓ Registros eliminados (inválidos): {len(df_lazy.collect()) - len(df_processed):,}")

print(f"\n✓ Features: {df_processed.shape[1]}")
print(f"✓ Valores faltantes: {df_processed.null_count().sum_horizontal()}")
print(f"✓ Duplicados: {len(df_processed) - df_processed.unique().shape[0]}")

print(f"\n✓ Estadísticas de features normalizados:")
print(df_processed.describe())

VALIDACIÓN DE CALIDAD

✓ Registros originales: 1,503
✓ Registros después limpieza: 1,503
✓ Registros eliminados (inválidos): 0

✓ Features: 11
✓ Valores faltantes: shape: (1,)
Series: 'sum' [u32]
[
	0
]
✓ Duplicados: 1185

✓ Estadísticas de features normalizados:
shape: (9, 12)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ statistic ┆ Age_norma ┆ Feeling   ┆ Irritable ┆ … ┆ Feeling   ┆ Problems  ┆ Suicide   ┆ Feeling  │
│ ---       ┆ lized     ┆ sad or    ┆ towards   ┆   ┆ of guilt_ ┆ of        ┆ attempt_n ┆ sad or   │
│ str       ┆ ---       ┆ Tearful_n ┆ baby &    ┆   ┆ normalize ┆ bonding   ┆ ormalized ┆ Tearful_ │
│           ┆ f64       ┆ ormali…   ┆ partn…    ┆   ┆ d         ┆ with      ┆ ---       ┆ encoded  │
│           ┆           ┆ ---       ┆ ---       ┆   ┆ ---       ┆ baby_…    ┆ f64       ┆ ---      │
│           ┆           ┆ f64       ┆ f64       ┆   ┆ f64       ┆ ---       ┆           ┆ f64      │
│           ┆ 

In [16]:
print("GUARDAR EN PARQUET")

output_file = 'data/postpartum_depression_processed.parquet'

print(f"\nGuardando a Parquet...")
print(f"Compresión: snappy (balance velocidad/tamaño)")

start_time = time.time()
df_processed.write_parquet(
    output_file,
    compression='snappy'
)
elapsed_time = time.time() - start_time

parquet_size = os.path.getsize(output_file) / (1024**2)
csv_size = os.path.getsize(csv_path) / (1024**2)
compression_ratio = csv_size / parquet_size

print(f"\nArchivo guardado")
print(f"Ubicación: {output_file}")
print(f"Tamaño CSV: {csv_size:.2f} MB")
print(f"Tamaño Parquet: {parquet_size:.2f} MB")
print(f"Compresión: {compression_ratio:.1f}x más pequeño")
print(f"Tiempo escritura: {elapsed_time:.2f}s")

GUARDAR EN PARQUET

Guardando a Parquet...
Compresión: snappy (balance velocidad/tamaño)

Archivo guardado
Ubicación: data/postpartum_depression_processed.parquet
Tamaño CSV: 0.11 MB
Tamaño Parquet: 0.02 MB
Compresión: 4.4x más pequeño
Tiempo escritura: 0.03s


##Descargar resultado

In [17]:
from google.colab import files

print("DESCARGAR ARCHIVO PROCESADO")

print(f"Preparando descarga...")
files.download('data/postpartum_depression_processed.parquet')

print(f"\nDescarga iniciada")
print(f"Archivo: postpartum_depression_processed.parquet")
print(f"Tamaño: {parquet_size:.2f} MB")
print(f"Formato: Parquet (100x más rápido que CSV)")

DESCARGAR ARCHIVO PROCESADO
Preparando descarga...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Descarga iniciada
Archivo: postpartum_depression_processed.parquet
Tamaño: 0.02 MB
Formato: Parquet (100x más rápido que CSV)


DATA AUMENTATION